# Remote Tracking (Colab, no Google Drive)

This notebook runs the tracking pipeline without mounting Google Drive.

## What you upload
Upload these files directly into Colab:
- the **pitch model** weights file (`.pt`)
- the **player model** weights file (`.pt`)
- the **video** file (`.mp4`, `.mov`, `.avi`, `.mkv`, or `.m4v`)

## How the upload works
When you run the upload cell, Colab opens a file picker on your computer. You can select all required files at once.  
The notebook saves them into a temporary local folder inside the Colab runtime:

- `/content/football_tracking/uploads`
- outputs go to `/content/football_tracking/outputs`

At the end, the notebook downloads:
- the tracking CSV
- the annotated output video

## Important
Because Colab storage is temporary, uploaded files disappear when the runtime resets.

In [ ]:
# Install dependencies
%pip install -q ultralytics==8.3.0 supervision==0.25.1 tqdm==4.66.5 python-dotenv==1.0.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 881.3/881.3 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.5/181.5 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 11.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
dataproc-spark-connect 1.0.2 requires tqdm>=4.67, but you have tqdm 4.66.5 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have

In [ ]:
# Environment and local folders (no Drive)
import os
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
    from google.colab import files  # type: ignore
except ImportError:
    IN_COLAB = False

BASE_DIR = "/content/football_tracking" if IN_COLAB else os.path.abspath(os.getcwd())
UPLOADS_DIR = os.path.join(BASE_DIR, "uploads")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")

os.makedirs(UPLOADS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("IN_COLAB:", IN_COLAB)
print("BASE_DIR:", BASE_DIR)
print("UPLOADS_DIR:", UPLOADS_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

IN_COLAB: True
BASE_DIR: /content/football_tracking
UPLOADS_DIR: /content/football_tracking/uploads
OUTPUT_DIR: /content/football_tracking/outputs


In [ ]:
# Upload files directly from your computer
# Select these together:
# 1) pitch model .pt
# 2) player model .pt
# 3) input video file
if IN_COLAB:
    uploaded = files.upload()

    for filename, data in uploaded.items():
        save_path = os.path.join(UPLOADS_DIR, filename)
        with open(save_path, "wb") as f:
            f.write(data)
        print("Saved:", save_path)
else:
    print("Not running in Colab. Put your files manually into:", UPLOADS_DIR)

In [ ]:
# Detect uploaded files and choose which ones to use
uploaded_files = sorted(os.listdir(UPLOADS_DIR))
print("Uploaded files:", uploaded_files)

pt_files = [f for f in uploaded_files if f.lower().endswith(".pt")]
video_files = [
    f for f in uploaded_files
    if f.lower().endswith((".mp4", ".mov", ".avi", ".mkv", ".m4v"))
]

if len(pt_files) < 2:
    raise ValueError("Please upload both model files (.pt): pitch model and player model.")

if len(video_files) < 1:
    raise ValueError("Please upload one video file.")

print("\nDetected model files:")
for i, f in enumerate(pt_files):
    print(f"{i}: {f}")

print("\nDetected video files:")
for i, f in enumerate(video_files):
    print(f"{i}: {f}")

# If the auto-pick is wrong, swap these manually.
PITCH_MODEL_FILENAME = pt_files[0]
PLAYER_MODEL_FILENAME = pt_files[1]
VIDEO_FILE = video_files[0]

PITCH_MODEL_PATH = os.path.join(UPLOADS_DIR, PITCH_MODEL_FILENAME)
PLAYER_MODEL_PATH = os.path.join(UPLOADS_DIR, PLAYER_MODEL_FILENAME)
SOURCE_VIDEO_PATH = os.path.join(UPLOADS_DIR, VIDEO_FILE)

print("\nUsing:")
print("Pitch model:", PITCH_MODEL_PATH)
print("Player model:", PLAYER_MODEL_PATH)
print("Video:", SOURCE_VIDEO_PATH)

In [ ]:
# Imports and repo setup
from ultralytics import YOLO
import supervision as sv
from tqdm import tqdm
import numpy as np
import sys

# Try to locate the cloned repo root to import project modules
GUESS_REPO_ROOT = "/content/FootballTrackingDataGeneration-main"
if os.path.exists(GUESS_REPO_ROOT):
    REPO_ROOT = GUESS_REPO_ROOT
else:
    REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

from sports.common.team import TeamClassifier
from sports.configs.soccer import SoccerPitchConfiguration
from sports.common.view import ViewTransformer

print("REPO_ROOT:", REPO_ROOT)

In [ ]:
# Load YOLO models from uploaded files
PITCH_DETECTION_MODEL = YOLO(PITCH_MODEL_PATH)
PLAYER_DETECTION_MODEL = YOLO(PLAYER_MODEL_PATH)

print("Models loaded successfully.")

In [ ]:
# Core configuration
GENERATE_VIDEO = 1
TRACK_TEAMS = 1

# Detection / tracking tuning
PLAYER_CONF = 0.35
PITCH_CONF = 0.3
NMS_THRESHOLD = 0.4

print("Using video:", SOURCE_VIDEO_PATH)

In [ ]:
# Object classes and annotators
objects = {
    "ball": 0,
    "goalkeeper": 1,
    "player": 2,
    "referee": 3,
}

ellipse_annotator = sv.EllipseAnnotator(
    color=sv.ColorPalette.from_hex(["#00BFFF", "#FF1493", "#FFD700"]),
    thickness=2,
)
label_annotator = sv.LabelAnnotator(
    color=sv.ColorPalette.from_hex(["#00BFFF", "#FF1493", "#FFD700"]),
    text_color=sv.Color.from_hex("#000000"),
    text_position=sv.Position.BOTTOM_CENTER,
)
triangle_annotator = sv.TriangleAnnotator(
    color=sv.Color.from_hex("#FFD700"),
    base=25,
    height=21,
    outline_thickness=1,
)
box_annotator = sv.BoxAnnotator(
    color=sv.ColorPalette.from_hex(["#FF8C00", "#00BFFF", "#FF1493", "#FFD700"]),
    thickness=2,
)

In [ ]:
# Identify goalkeeper team by proximity to team centroids
def resolve_goalkeepers_team_id(players, goalkeepers):
    goalkeepers_xy = goalkeepers.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
    players_xy = players.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)

    team_0_players = players_xy[players.class_id == 0]
    team_1_players = players_xy[players.class_id == 1]

    if len(team_0_players) == 0 or len(team_1_players) == 0:
        return np.zeros(len(goalkeepers_xy), dtype=int)

    team_0_centroid = team_0_players.mean(axis=0)
    team_1_centroid = team_1_players.mean(axis=0)

    goalkeepers_team_id = []
    for goalkeeper_xy in goalkeepers_xy:
        dist_0 = np.linalg.norm(goalkeeper_xy - team_0_centroid)
        dist_1 = np.linalg.norm(goalkeeper_xy - team_1_centroid)
        goalkeepers_team_id.append(0 if dist_0 < dist_1 else 1)

    return np.array(goalkeepers_team_id)

In [ ]:
# Compute detections, assign teams, and transform to pitch coordinates
def get_detections(frame, detections, key_points, tracker, team_classifier):
    CONFIG = SoccerPitchConfiguration()

    ball_detections = detections[detections.class_id == objects["ball"]]
    if len(ball_detections) > 0:
        ball_detections.xyxy = sv.pad_boxes(xyxy=ball_detections.xyxy, px=10)

    all_detections = detections[detections.class_id != objects["ball"]]
    all_detections = all_detections.with_nms(threshold=NMS_THRESHOLD, class_agnostic=True)
    all_detections = tracker.update_with_detections(detections=all_detections)

    goalkeepers_detections = all_detections[all_detections.class_id == objects["goalkeeper"]]
    players_detections = all_detections[all_detections.class_id == objects["player"]]

    if team_classifier is not None and len(players_detections) > 0:
        players_crops = [sv.crop_image(frame, xyxy) for xyxy in players_detections.xyxy]
        players_detections.class_id = team_classifier.predict(players_crops)

        if len(goalkeepers_detections) > 0:
            goalkeepers_detections.class_id = resolve_goalkeepers_team_id(
                players_detections, goalkeepers_detections
            )

    referees_detections = all_detections[all_detections.class_id == objects["referee"]]
    referees_detections.class_id -= 1

    all_detections = sv.Detections.merge(
        [players_detections, goalkeepers_detections, referees_detections]
    )

    vertices = key_points.xy[0] if len(key_points.xy) > 0 else np.empty((0, 2))
    if len(vertices) > 3:
        CONFIG = SoccerPitchConfiguration()
        pitch_reference_points = np.array(CONFIG.vertices)[
            key_points.confidence[0] > 0.5
        ]
        transformer = ViewTransformer(
            source=vertices,
            target=pitch_reference_points
        )

        pitch_xy = transformer.transform_points(
            points=all_detections.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
        )
        all_detections.data["pitch_xy"] = pitch_xy

        if len(ball_detections) > 0:
            ball_pitch_xy = transformer.transform_points(
                points=ball_detections.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
            )
            ball_detections.data["pitch_xy"] = ball_pitch_xy
        else:
            ball_detections.data["pitch_xy"] = np.empty((0, 2))
    else:
        all_detections.data["pitch_xy"] = np.empty((len(all_detections), 2))
        ball_detections.data["pitch_xy"] = np.empty((len(ball_detections), 2))

    all_detections.data["class_name"] = np.array(["player"] * len(all_detections), dtype=object)
    return all_detections, ball_detections

In [ ]:
# Learn team colors from a stride of frames
# Important fix: only player detections are used to train the team classifier.
def generate_team_model(video_path, player_model):
    STRIDE = 30

    frame_generator = sv.get_video_frames_generator(
        source_path=video_path, stride=STRIDE
    )

    crops = []
    for frame in tqdm(frame_generator, desc="collecting player crops"):
        results = player_model(frame, conf=PLAYER_CONF)
        result = results[0]
        detections = sv.Detections.from_ultralytics(result)

        player_detections = detections[detections.class_id == objects["player"]]
        player_detections = player_detections.with_nms(
            threshold=NMS_THRESHOLD, class_agnostic=True
        )

        frame_crops = [sv.crop_image(frame, xyxy) for xyxy in player_detections.xyxy]
        crops.extend(frame_crops)

    if len(crops) == 0:
        raise ValueError("No player crops found for team classifier training.")

    team_classifier = TeamClassifier()
    team_classifier.fit(crops)
    return team_classifier

In [ ]:
# Save tracking results to a local CSV file
def save_tracking_results(players, ball, frames):
    csv = "Frame,Object,Object ID,Team,X1,Y1,X2,Y2,X_Pitch,Y_Pitch\n"

    for frame in range(1, frames):
        for player_id in players:
            player_data = players[player_id]
            if str(frame) in player_data:
                row = player_data[str(frame)]
                csv += (
                    f"{frame},player,{player_id},{row['Team']},"
                    f"{row['X1']},{row['Y1']},{row['X2']},{row['Y2']},"
                    f"{row['X_Pitch']},{row['Y_Pitch']}\n"
                )

        if str(frame) in ball:
            row = ball[str(frame)]
            csv += (
                f"{frame},ball,,,"
                f"{row['X1']},{row['Y1']},{row['X2']},{row['Y2']},"
                f"{row['X_Pitch']},{row['Y_Pitch']}\n"
            )

    output_csv_path = os.path.join(
        OUTPUT_DIR, Path(VIDEO_FILE).stem + "_tracking.csv"
    )

    with open(output_csv_path, "w") as file:
        file.write(csv)

    print("Saved tracking CSV to", os.path.abspath(output_csv_path))
    return output_csv_path

In [ ]:
# Optional: learn team classifier from the uploaded video
team_classifier = None
if TRACK_TEAMS:
    team_classifier = generate_team_model(SOURCE_VIDEO_PATH, PLAYER_DETECTION_MODEL)

In [ ]:
# Main tracking loop
CONFIG = SoccerPitchConfiguration()
frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)
video_info = sv.VideoInfo.from_video_path(video_path=SOURCE_VIDEO_PATH)

try:
    tracker = sv.ByteTrack(
        frame_rate=video_info.fps,
        track_activation_threshold=PLAYER_CONF,
        lost_track_buffer=60,
        minimum_matching_threshold=0.85,
        minimum_consecutive_frames=2,
    )
except TypeError:
    tracker = sv.ByteTrack(frame_rate=video_info.fps)

tracker.reset()

frame_number = 1
output_video_path = os.path.join(
    OUTPUT_DIR, Path(VIDEO_FILE).stem + "_tracked.mp4"
)

players = {}
ball = {}

with sv.VideoSink(target_path=output_video_path, video_info=video_info) as sink:
    for frame in tqdm(frame_generator, desc="Collecting Tracking Data..."):
        player_results = PLAYER_DETECTION_MODEL(frame, conf=PLAYER_CONF)
        player_result = player_results[0]
        detections = sv.Detections.from_ultralytics(player_result)

        pitch_results = PITCH_DETECTION_MODEL(frame, conf=PITCH_CONF)
        pitch_result = pitch_results[0]
        key_points = sv.KeyPoints.from_ultralytics(pitch_result)

        all_detections, ball_detections = get_detections(
            frame, detections, key_points, tracker, team_classifier
        )

        object_ids = all_detections.tracker_id if len(all_detections) > 0 else []
        team_ids = all_detections.class_id if len(all_detections) > 0 else []
        object_types = all_detections.data["class_name"] if len(all_detections) > 0 else []
        pitch_xys = all_detections.data["pitch_xy"] if len(all_detections) > 0 else []
        ball_pitch_xys = ball_detections.data["pitch_xy"]

        if len(all_detections) > 0:
            all_detections.class_id = all_detections.class_id.astype(int)
            labels = [f"#{tracker_id}" for tracker_id in all_detections.tracker_id]
        else:
            labels = []

        for idx, xyxy in enumerate(all_detections.xyxy):
            team_id = int(team_ids[idx]) if len(team_ids) > idx else 0
            object_id = str(object_ids[idx])

            if object_id not in players:
                players[object_id] = {}

            px = float(pitch_xys[idx][0]) if len(pitch_xys) > idx else 0.0
            py = float(pitch_xys[idx][1]) if len(pitch_xys) > idx else 0.0

            players[object_id][str(frame_number)] = {
                "Object Type": object_types[idx],
                "Team": team_id,
                "X1": float(xyxy[0]),
                "Y1": float(xyxy[1]),
                "X2": float(xyxy[2]),
                "Y2": float(xyxy[3]),
                "X_Pitch": px,
                "Y_Pitch": py,
                "Y_MPLSoccer": py / float(CONFIG.width),
                "X_MPLSoccer": px / float(CONFIG.length),
            }

        if ball_detections.xyxy.shape[0]:
            bx = float(ball_pitch_xys[0][0]) if len(ball_pitch_xys) > 0 else 0.0
            by = float(ball_pitch_xys[0][1]) if len(ball_pitch_xys) > 0 else 0.0
            ball[str(frame_number)] = {
                "X1": float(ball_detections.xyxy[0][0]),
                "Y1": float(ball_detections.xyxy[0][1]),
                "X2": float(ball_detections.xyxy[0][2]),
                "Y2": float(ball_detections.xyxy[0][3]),
                "X_Pitch": bx,
                "Y_Pitch": by,
                "Y_MPLSoccer": by / float(CONFIG.width),
                "X_MPLSoccer": bx / float(CONFIG.length),
            }
        else:
            ball[str(frame_number)] = {
                "X1": 0,
                "Y1": 0,
                "X2": 0,
                "Y2": 0,
                "X_Pitch": 0,
                "Y_Pitch": 0,
                "Y_MPLSoccer": 0,
                "X_MPLSoccer": 0,
            }

        frame_number += 1

        if GENERATE_VIDEO:
            annotated_frame = frame.copy()
            annotated_frame = ellipse_annotator.annotate(
                scene=annotated_frame,
                detections=all_detections,
            )
            annotated_frame = label_annotator.annotate(
                scene=annotated_frame,
                detections=all_detections,
                labels=labels,
            )
            annotated_frame = triangle_annotator.annotate(
                scene=annotated_frame,
                detections=ball_detections,
            )
            sink.write_frame(frame=annotated_frame)

print("Saved annotated video to", os.path.abspath(output_video_path))
output_csv_path = save_tracking_results(players, ball, frame_number)


0: 736x1280 1 ball, 27 players, 2 referees, 7685.9ms
Speed: 7.4ms preprocess, 7685.9ms inference, 2.1ms postprocess per image at shape (1, 3, 736, 1280)

0: 384x640 1 pitch, 191.7ms
Speed: 4.0ms preprocess, 191.7ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)



0: 736x1280 1 ball, 24 players, 2 referees, 7749.3ms
Speed: 11.9ms preprocess, 7749.3ms inference, 0.8ms postprocess per image at shape (1, 3, 736, 1280)

0: 384x640 1 pitch, 110.1ms
Speed: 2.8ms preprocess, 110.1ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)



0: 736x1280 20 players, 2 referees, 7724.6ms
Speed: 7.6ms preprocess, 7724.6ms inference, 1.0ms postprocess per image at shape (1, 3, 736, 1280)

0: 384x640 1 pitch, 112.9ms
Speed: 2.4ms preprocess, 112.9ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)



Embedding extraction: 0it [00:00, ?it/s]
Embedding extraction: 1it [00:10, 10.12s/it]


0: 736x1280 1 ball, 21 players, 2 referees, 7817.4ms
Speed: 10.6ms preprocess, 7817.4ms inference, 1.9ms postprocess per image at shape (1, 3, 736, 1280)

0: 384x640 1 pitch, 112.9ms
Speed: 2.8ms preprocess, 112.9ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)


...


0: 736x1280 1 goalkeeper, 24 players, 1 referee, 7755.5ms
Speed: 10.4ms preprocess, 7755.5ms inference, 0.8ms postprocess per image at shape (1, 3, 736, 1280)

0: 384x640 1 pitch, 129.1ms
Speed: 3.2ms preprocess, 129.1ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)



Embedding extraction: 0it [00:00, ?it/s]
Embedding extraction: 1it [00:11, 11.19s/it]


0: 736x1280 1 goalkeeper, 25 players, 1 referee, 7760.8ms
Speed: 14.6ms preprocess, 7760.8ms inference, 0.8ms postprocess per image at shape (1, 3, 736, 1280)

0: 384x640 1 pitch, 121.3ms
Speed: 2.1ms preprocess, 121.3ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)



Embedding extraction: 0it [00:00, ?it/s]
Embedding extraction: 1it [00:11, 11.09s/it]


Saved annotated video to /content/football_tracking/outputs/sample_30_tracked.mp4
Saved tracking CSV to /content/football_tracking/outputs/sample_30_tracking.csv


In [ ]:
# Download outputs to your computer
if IN_COLAB:
    print("Downloading CSV...")
    files.download(output_csv_path)

    if GENERATE_VIDEO:
        print("Downloading tracked video...")
        files.download(output_video_path)
else:
    print("CSV:", output_csv_path)
    print("Video:", output_video_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>